In [14]:
import pandas as pd
import numpy as np

from pathlib import Path

OUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/output")
parquet_path = OUT_DIR / "ait_ads_wazuh.parquet"

df = pd.read_parquet(parquet_path)

print("Loaded:", df.shape)
df.head(3)

Loaded: (2600263, 14)


,file,timestamp,location,agent_id,agent_name,agent_ip,hostname,program,full_log,rule_id,rule_description,rule_groups,rule_level,rule_groups_str
0,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
1,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
2,fox_wazuh.json,2022-01-15 02:32:37+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"


In [15]:
import ipaddress

# timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df = df.dropna(subset=["timestamp"]).copy()

# y_priority
def map_priority(level):
    if pd.isna(level):
        return None
    level = float(level)
    if level <= 3: return "low"
    if level <= 6: return "medium"
    if level <= 10: return "high"
    return "critical"

df["y_priority"] = df["rule_level"].apply(map_priority)

# internal/external IP
df["agent_ip"] = df["agent_ip"].astype(str)

def is_internal_ip(ip):
    try:
        return int(ipaddress.ip_address(ip).is_private)
    except:
        return 0

df["is_internal_ip"] = df["agent_ip"].apply(is_internal_ip).astype("int8")

df["y_priority"].value_counts()

y_priority
medium      1873973
low          592722
high         131901
critical       1667
Name: count, dtype: int64

In [16]:
df["hour"] = df["timestamp"].dt.hour.astype("int16")
df["dow"] = df["timestamp"].dt.dayofweek.astype("int16")
df["is_weekend"] = df["dow"].isin([5,6]).astype("int8")
df["is_night"] = (df["hour"] < 6).astype("int8")

In [17]:
df = df.sort_values(["agent_ip", "timestamp"]).reset_index(drop=True)

tmp = (
    df.groupby("agent_ip")
      .rolling("5min", on="timestamp")["rule_id"]
      .count()
      .reset_index(level=0, drop=True)
)

df["ip_burst_5m"] = tmp.to_numpy(dtype=np.int16)
df["log_ip_burst_5m"] = np.log1p(df["ip_burst_5m"]).astype("float32")

df[["agent_ip","timestamp","ip_burst_5m"]].head()

/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,agent_ip,timestamp,ip_burst_5m
0,10.132.56.1,2022-01-19 01:02:04.669876+00:00,1
1,10.132.56.1,2022-01-19 01:02:04.669876+00:00,2
2,10.132.56.1,2022-01-19 01:02:04.676411+00:00,3
3,10.132.56.1,2022-01-19 01:02:04.676411+00:00,4
4,10.132.56.1,2022-01-19 01:02:04.704695+00:00,5


In [18]:
df = df.sort_values(["rule_id", "timestamp"]).reset_index(drop=True)

tmp_rule = (
    df.groupby("rule_id")
      .rolling("5min", on="timestamp")["agent_ip"]
      .count()
      .reset_index(level=0, drop=True)
)

df["rule_burst_5m"] = tmp_rule.to_numpy(dtype=np.int16)
df["log_rule_burst_5m"] = np.log1p(df["rule_burst_5m"]).astype("float32")

df[["rule_id","rule_burst_5m"]].head()

/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,rule_id,rule_burst_5m
0,20100,1
1,20100,2
2,20100,3
3,20100,4
4,20100,5


In [19]:
df["rule_groups_str"] = df["rule_groups_str"].fillna("").astype(str)

df["grp_attack"] = df["rule_groups_str"].str.contains("attack", case=False).astype(int)
df["grp_recon"] = df["rule_groups_str"].str.contains("recon", case=False).astype(int)
df["grp_scan"] = df["rule_groups_str"].str.contains("scan", case=False).astype(int)
df["grp_auth"] = df["rule_groups_str"].str.contains("authentication", case=False).astype(int)
df["grp_ids"] = df["rule_groups_str"].str.contains("ids", case=False).astype(int)

In [20]:
df["location"] = df["location"].fillna("").astype(str)

df["is_apache"] = df["location"].str.contains("apache", case=False).astype(int)
df["is_suricata"] = df["location"].str.contains("suricata", case=False).astype(int)
df["is_syslog"] = df["location"].str.contains("syslog", case=False).astype(int)
df["is_mail"] = df["location"].str.contains("mail", case=False).astype(int)

In [21]:
df["full_log"] = df["full_log"].fillna("").astype(str)

# HTTP коды
df["has_400"] = df["full_log"].str.contains(" 400 ").astype(int)
df["has_401"] = df["full_log"].str.contains(" 401 ").astype(int)
df["has_403"] = df["full_log"].str.contains(" 403 ").astype(int)
df["has_500"] = df["full_log"].str.contains(" 500 ").astype(int)

# HTTP методы
df["has_post"] = df["full_log"].str.contains("POST").astype(int)
df["has_get"] = df["full_log"].str.contains("GET").astype(int)

# Подозрительные паттерны
df["has_sql"] = df["full_log"].str.contains("select|union|sql", case=False, regex=True).astype(int)
df["has_admin"] = df["full_log"].str.contains("/admin", case=False).astype(int)
df["has_wp"] = df["full_log"].str.contains("wp-", case=False).astype(int)

In [22]:
# Цель
y = df["y_priority"].astype(str)

# Категориальные признаки (из того, что у тебя есть)
cat_cols = ["location", "program", "rule_groups_str", "agent_name", "hostname"]
cat_cols = [c for c in cat_cols if c in df.columns]

# Числовые признаки (эвристики)
num_cols = [
    "hour", "dow", "is_weekend", "is_night",
    "is_internal_ip",
    "log_ip_burst_5m", "log_rule_burst_5m",
    "grp_attack", "grp_recon", "grp_scan", "grp_auth", "grp_ids",
    "is_apache", "is_suricata", "is_syslog", "is_mail",
    "has_400", "has_401", "has_403", "has_500",
    "has_post", "has_get",
    "has_sql", "has_admin", "has_wp"
]

# Оставим только реально существующие
num_cols = [c for c in num_cols if c in df.columns]

# X
X = df[cat_cols + num_cols].copy()

# Чистим категории
for c in cat_cols:
    X[c] = X[c].fillna("NA").astype(str)

# Чистим числа
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)

print("X shape:", X.shape)
print("cat_cols:", cat_cols)
print("num_cols:", num_cols)
print("y dist:\n", y.value_counts())
X.head(3)

X shape: (2600263, 30)
cat_cols: ['location', 'program', 'rule_groups_str', 'agent_name', 'hostname']
num_cols: ['hour', 'dow', 'is_weekend', 'is_night', 'is_internal_ip', 'log_ip_burst_5m', 'log_rule_burst_5m', 'grp_attack', 'grp_recon', 'grp_scan', 'grp_auth', 'grp_ids', 'is_apache', 'is_suricata', 'is_syslog', 'is_mail', 'has_400', 'has_401', 'has_403', 'has_500', 'has_post', 'has_get', 'has_sql', 'has_admin', 'has_wp']
y dist:
 y_priority
medium      1873973
low          592722
high         131901
critical       1667
Name: count, dtype: int64


,location,program,rule_groups_str,agent_name,hostname,hour,dow,is_weekend,is_night,is_internal_ip,...,is_mail,has_400,has_401,has_403,has_500,has_post,has_get,has_sql,has_admin,has_wp
0,/var/log/suricata/fast.log,NA,"ids,fts",wazuh-client,NA,0,4,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,/var/log/suricata/fast.log,NA,"ids,fts",wazuh-client,NA,0,4,0,1,1,...,0,0,0,0,0,0,0,0,0,0
2,/var/log/suricata/fast.log,NA,"ids,fts",wazuh-client,NA,0,4,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train dist:\n", y_train.value_counts(normalize=True))
print("Test dist:\n", y_test.value_counts(normalize=True))

Train: (2080210, 30) Test: (520053, 30)
Train dist:
 y_priority
medium      0.720686
low         0.227947
high        0.050726
critical    0.000641
Name: proportion, dtype: float64
Test dist:
 y_priority
medium      0.720686
low         0.227948
high        0.050726
critical    0.000640
Name: proportion, dtype: float64


In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
)

gb = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=400,
    max_depth=None,
    min_samples_leaf=30,
    l2_regularization=1e-6,
    random_state=42
)

model_gb = Pipeline(steps=[
    ("prep", preprocess),
    ("gb", gb)
])

# Веса классов (компенсация дисбаланса)
sw = compute_sample_weight(class_weight="balanced", y=y_train)

model_gb.fit(X_train, y_train, gb__sample_weight=sw)
print("Trained ✅")

Trained ✅


In [25]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd

pred = model_gb.predict(X_test)

print(classification_report(y_test, pred, digits=3))
print("macro-F1:", f1_score(y_test, pred, average="macro"))

labels = ["low", "medium", "high", "critical"]
cm = confusion_matrix(y_test, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_df

              precision    recall  f1-score   support

    critical      0.277     0.967     0.431       333
        high      0.998     0.969     0.983     26380
         low      1.000     1.000     1.000    118545
      medium      1.000     1.000     1.000    374795

    accuracy                          0.998    520053
   macro avg      0.819     0.984     0.853    520053
weighted avg      0.999     0.998     0.999    520053

macro-F1: 0.8534117299577135


,pred_low,pred_medium,pred_high,pred_critical
true_low,118544,0,1,0
true_medium,4,374704,52,35
true_high,1,12,25562,805
true_critical,0,0,11,322


In [26]:
err = X_test.copy()
err["y_true"] = y_test.values
err["y_pred"] = pred

bad = err[err["y_true"] != err["y_pred"]].copy()
print("Ошибок:", bad.shape[0])

# ошибки по важным классам
bad_hc = bad[bad["y_true"].isin(["high", "critical"])].copy()
print("Ошибок high/critical:", bad_hc.shape[0])

bad_hc.head(30)

Ошибок: 921
Ошибок high/critical: 829


,location,program,rule_groups_str,agent_name,hostname,hour,dow,is_weekend,is_night,is_internal_ip,...,has_401,has_403,has_500,has_post,has_get,has_sql,has_admin,has_wp,y_true,y_pred
296649,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,10,5,1,0,1,...,0,0,0,0,0,0,0,0,high,critical
299502,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,8,0,0,0,1,...,0,0,0,0,0,0,0,0,high,critical
301058,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,15,5,1,0,1,...,0,0,0,0,0,0,0,0,high,critical
299795,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,16,3,0,0,1,...,0,0,0,0,0,0,0,0,high,critical
299819,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,7,4,0,0,1,...,0,0,0,0,0,0,0,0,high,critical
297096,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,18,3,0,0,1,...,0,0,0,0,0,0,0,0,high,critical
297612,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,14,6,1,0,1,...,0,0,0,0,0,0,0,0,high,critical
297419,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,13,5,1,0,1,...,0,0,0,0,0,0,0,0,high,critical
298268,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,10,4,0,0,1,...,0,0,0,0,0,0,0,0,high,critical
296543,/var/log/suricata/fast.log,NA,ids,wazuh-client,NA,13,4,0,0,1,...,0,0,0,0,0,0,0,0,high,critical
